# Démo 4 · Tout mettre ensemble : le Titanic

Le 15 avril 1912, le *Titanic* a sombré après avoir heurté un iceberg ; plus de 1 500 des
2 224 passagers et membres d’équipage ont péri. La
[compétition Titanic](https://www.kaggle.com/c/titanic) de Kaggle pose une question simple :
**à partir de ce que l’on sait d’un passager, peut-on prédire s’il a survécu ?**

Ce notebook parcourt un projet complet de science des données autour de cette question, en
réutilisant les outils de la démo 1, du
[notebook 01](01_validation_and_hyperparameter_search.ipynb) et du
[notebook 02](02_bootstrap_confidence_intervals.ipynb) :

```text
question → données → exploration → nettoyage et création de variables → modèle → validation → prédiction et partage
```

À la fin, vous serez capables de :

- Inspecter un nouveau jeu de données : ses colonnes, sa cible, ses valeurs manquantes.
- Utiliser des graphiques pour repérer les variables liées à la cible.
- Transformer des colonnes brutes (texte, catégories, valeurs manquantes) en variables numériques dans une fonction réutilisable.
- Comparer plusieurs classifieurs, les combiner dans un ensemble et valider le résultat par validation croisée.
- Produire un fichier de prédictions au format attendu par Kaggle.

### Exercices
Six exercices sont placés là où ils s’intègrent dans le projet. Chacun comporte une cellule
vide pour votre travail et une réponse de référence à ouvrir **après** avoir essayé.

| | Exercice | Section |
|---|---|---|
| Q1 | Imputer les âges manquants | D · Variable 4 |
| Q2 | Créer des variables à partir de `Fare`, `Name` et `Ticket` | D · Variables 7 à 9 |
| Q3 | Essayer la PCA sur les variables | E |
| Q4 | Ajuster un modèle avec `RandomizedSearchCV` | I |
| Q5 | Atteindre la meilleure accuracy possible | I |
| Q6 | Un intervalle de confiance bootstrap pour le modèle final | I |

## Configuration

In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "numpy>=1.26",
        "pandas>=2.2,<4",
        "matplotlib>=3.7",
        "seaborn>=0.13,<0.14",
        "scikit-learn>=1.4",
        "scipy>=1.11",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    RandomForestClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/milarobotlearningcourse/data_science/main/demo_4/datasets/"

# Locally, find demo_4/datasets by walking up from the notebook folder.
ROOT, DATA_DIR = Path.cwd(), None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "demo_4" / "datasets").is_dir():
        ROOT, DATA_DIR = folder, folder / "demo_4" / "datasets"
        break


def data_source(filename):
    """Return the local CSV path if the repository is available, otherwise its URL."""
    if DATA_DIR is not None:
        return DATA_DIR / filename
    return DATA_URL + filename


print("Reading data from:", DATA_DIR or DATA_URL)

## A. Les données

Kaggle fournit deux fichiers :

- `titanic-train.csv` : 891 passagers **avec** la réponse (`Survived`). C’est à partir de ce fichier que nous apprenons.
- `titanic-test.csv` : 418 autres passagers **sans** la réponse. Kaggle garde leur sort
  secret et évalue les prédictions que nous soumettons.

Nous conservons les fichiers tels qu’ils ont été chargés dans `train_raw` et `test_raw` et ne
les modifions jamais : nous pouvons ainsi toujours repartir des données d’origine.

In [ ]:
train_raw = pd.read_csv(data_source("titanic-train.csv"))
test_raw = pd.read_csv(data_source("titanic-test.csv"))
print("train:", train_raw.shape, "| test:", test_raw.shape)
train_raw.info()

In [ ]:
display(train_raw.head())

### Signification des colonnes

| Colonne | Signification |
|---|---|
| `PassengerId` | Identifiant de la ligne |
| `Survived` | 1 = a survécu, 0 = a péri (**la cible**, absente du fichier de test) |
| `Pclass` | Classe du billet : 1 = première (supérieure), 2 = deuxième (moyenne), 3 = troisième (inférieure) ; un indicateur indirect du statut socio-économique |
| `Name` | Nom complet, avec un titre comme *Mr*, *Mrs*, *Miss* ou *Master* |
| `Sex` | `male` ou `female` |
| `Age` | Âge en années ; fractionnaire sous 1 an, et de la forme *xx.5* lorsqu’il est estimé |
| `SibSp` | Nombre de frères, de sœurs ou de conjoints à bord |
| `Parch` | Nombre de parents ou d’enfants à bord (certains enfants voyageaient seulement avec une nourrice, d’où `Parch` = 0) |
| `Ticket` | Numéro du billet |
| `Fare` | Prix du billet (en livres sterling de 1912) |
| `Cabin` | Numéro de cabine, souvent manquant |
| `Embarked` | Port d’embarquement : C = Cherbourg, Q = Queenstown, S = Southampton |

**Vérifiez votre compréhension :** combien de variables explicatives pourrions-nous utiliser,
et quelle colonne est la cible ?

<details>
<summary>Réponse</summary>

`Survived` est la cible. `PassengerId` n’est qu’un identifiant : il n’apporte aucune
information sur la survie. Il reste donc **10 variables candidates** : `Pclass`, `Name`, `Sex`,
`Age`, `SibSp`, `Parch`, `Ticket`, `Fare`, `Cabin` et `Embarked`. Plusieurs d’entre elles
(`Name`, `Ticket`, `Cabin`) sont du texte et doivent être retravaillées avant qu’un modèle
puisse les utiliser.

</details>

## B. Valeurs manquantes

Les modèles de scikit-learn n’acceptent pas les valeurs manquantes : il faut donc savoir où
elles se trouvent, dans les **deux** fichiers.

In [ ]:
missing = pd.DataFrame({
    "train": train_raw.isna().sum(),
    "test": test_raw.isna().sum(),
})
display(missing[missing.sum(axis=1) > 0])

- `Age` manque pour environ 20 % des passagers : il vaut la peine de compléter ces valeurs avec soin (exercice Q1).
- `Cabin` manque pour environ 77 % des passagers : c’est trop pour les compléter, mais le fait *d’avoir* une cabine enregistrée peut être informatif en soi.
- `Embarked` manque deux fois dans le fichier d’entraînement ; nous utiliserons le port le plus fréquent.
- `Fare` manque une fois, **uniquement dans le fichier de test**. Toute variable construite à partir de `Fare` doit gérer cette ligne, sinon la prédiction sur le fichier de test échouera.

## C. Explorer les données

### Combien de passagers ont survécu ?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=train_raw, x="Survived", ax=axes[0])
axes[0].set_title("Number of passengers")
(train_raw["Survived"].value_counts(normalize=True).sort_index() * 100).plot(kind="bar", ax=axes[1])
axes[1].set_title("Share of passengers (%)")
axes[1].set_xlabel("Survived")
plt.tight_layout()
plt.show()

Environ 38 % des passagers du fichier d’entraînement ont survécu. Un modèle qui prédirait
toujours « a péri » aurait donc raison 62 % du temps : **chaque modèle doit battre cette
baseline**.

### L’âge et la survie

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=train_raw, x="Age", hue="Survived", bins=20, ax=axes[0])
axes[0].set_title("Age distribution (counts)")
sns.kdeplot(data=train_raw, x="Age", hue="Survived", fill=True, common_norm=False, ax=axes[1])
axes[1].set_title("Age distribution (density within each group)")
plt.tight_layout()
plt.show()

Le graphique de densité remet chaque groupe à l’échelle pour que l’on puisse comparer leurs
formes. Les jeunes enfants sont plus fréquents parmi les survivants ; la plupart des autres
âges se ressemblent dans les deux groupes.

### Valeurs aberrantes
Les boîtes à moustaches montrent la médiane, les 50 % de valeurs centrales (la boîte) et les
points éloignés du reste.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=train_raw, x="Survived", y="Age", hue="Sex", ax=axes[0])
sns.boxplot(data=train_raw, x="Survived", y="Fare", hue="Sex", ax=axes[1])
plt.tight_layout()
plt.show()

Quelques tarifs dépassent 500 £ alors que la médiane est d’environ 14 £. Une variable aussi
asymétrique est souvent plus facile à exploiter pour les modèles après une transformation
logarithmique (une idée pour Q2).

### Corrélations entre les colonnes numériques
`numeric_only=True` ignore les colonnes textuelles (`Name`, `Sex`, …), pour lesquelles une
corrélation n’a pas de sens.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(train_raw.drop(columns="PassengerId").corr(numeric_only=True),
            vmin=-1, vmax=1, cmap="coolwarm", annot=True, fmt=".2f", square=True)
plt.show()

`Pclass` (−0,34) et `Fare` (+0,26) sont les colonnes numériques les plus liées à `Survived`.
La corrélation ne capte que les relations linéaires, et `Sex` n’est même pas encore inclus :
c’est donc un point de départ, pas un verdict.

## D. Création de variables (*feature engineering*)

Nous examinons maintenant chaque colonne et décidons comment la transformer en variable
numérique. Les transformations de cette section sont des **explorations** faites sur une
copie appelée `eda` ; la section E regroupe celles que nous conservons dans une seule
fonction, que nous appliquons aux deux fichiers.

Les diagrammes en barres ci-dessous montrent le **taux** de survie par groupe. Le trait noir
sur chaque barre est un intervalle de confiance à 95 % que seaborn calcule **par bootstrap**,
comme dans le notebook 02.

In [ ]:
eda = train_raw.copy()

### Variable 1 : `Pclass`

In [ ]:
sns.barplot(data=eda, x="Pclass", y="Survived")
plt.ylabel("Survival rate")
plt.show()

63 % des passagers de première classe ont survécu, contre 47 % en deuxième classe et 24 % en
troisième.

`Pclass` est codée 1, 2, 3. Un modèle pourrait lire ces valeurs comme une quantité (« la
classe 3 vaut trois fois la classe 1 »). L’**encodage one-hot** évite ce problème : une
colonne 0/1 par classe.

In [ ]:
pclass_dummies = pd.get_dummies(eda["Pclass"], prefix="Pclass", dtype=int)
display(pclass_dummies.head())

### Variable 2 : `Sex`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(data=eda, x="Sex", y="Survived", ax=axes[0])
axes[0].set_ylabel("Survival rate")
sns.countplot(data=eda, x="Sex", hue="Survived", palette="RdBu", ax=axes[1])
plt.tight_layout()
plt.show()

C’est le signal le plus fort des données : 74 % des femmes ont survécu, contre 19 % des hommes
(« les femmes et les enfants d’abord »). Les modèles ont besoin de nombres : nous associons
donc les deux valeurs à 1 et 0 avec `map`. Comme `map` renvoie une **nouvelle** colonne, nous
la réassignons explicitement.

In [ ]:
eda["Sex"] = eda["Sex"].map({"male": 1, "female": 0})
display(eda["Sex"].value_counts())

### Variable 3 : `Cabin`
La première lettre d’un numéro de cabine indique le **pont**. Dans ce rapide aperçu, les
cabines manquantes sont affichées comme `"?"`.

In [ ]:
eda["Deck"] = eda["Cabin"].str[0].fillna("?")
sns.barplot(data=eda, x="Deck", y="Survived", order=sorted(eda["Deck"].unique()))
plt.ylabel("Survival rate")
plt.show()

Certains ponts comptent très peu de passagers (regardez la largeur de leurs intervalles de
confiance). Le motif le plus net est plus simple : les passagers **avec** une cabine
enregistrée ont survécu beaucoup plus souvent (67 %) que les autres (30 %), sans doute parce
que les cabines étaient surtout enregistrées pour les passagers de première classe. Nous
gardons une variable 0/1 : *une cabine est-elle enregistrée ?*

In [ ]:
eda["Cabin"] = eda["Cabin"].notna().astype(int)
sns.barplot(data=eda, x="Cabin", y="Survived")
plt.xlabel("Cabin recorded")
plt.ylabel("Survival rate")
plt.show()

### Variable 4 : `Age`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
bins = range(0, 82, 2)
sns.histplot(eda.loc[eda["Survived"] == 1, "Age"], bins=bins, kde=True, color="tab:blue", ax=axes[0])
axes[0].set_title("Survived")
sns.histplot(eda.loc[eda["Survived"] == 0, "Age"], bins=bins, kde=True, color="tab:red", ax=axes[1])
axes[1].set_title("Died")
plt.tight_layout()
plt.show()

La relation entre l’âge et la survie n’est pas linéaire : les plus jeunes enfants s’en sont
le mieux sortis, et l’effet s’estompe rapidement. Des **tranches d’âge** permettent à un
modèle de capter ce phénomène. Mais il faut d’abord décider quoi faire des 177 âges manquants.

### Exercice Q1 · Imputation
La cellule ci-dessous remplace chaque âge manquant par la valeur provisoire **−0,5**, qui
tombe ensuite dans sa propre tranche d’âge « inconnu ». On conserve ainsi l’information
« l’âge était manquant », mais on écarte tout ce que l’on sait par ailleurs sur ces passagers.

**Votre tâche :** remplacez cette valeur provisoire par une meilleure estimation. L’âge moyen
est un début ; mieux encore, le **titre** contenu dans `Name` (*Master* désignait les jeunes
garçons, *Miss* les femmes non mariées, *Mrs* les femmes mariées) en dit long sur l’âge d’un
passager. Réfléchissez ensuite à la manière d’intégrer votre méthode dans la fonction
`preprocess` de la section E.

In [ ]:
## Your turn (Q1): replace this with a better technique.
eda["Age"] = eda["Age"].fillna(-0.5)

<details>
<summary>Réponse de référence Q1 — à ouvrir après avoir essayé</summary>

Extrayez le titre, puis remplacez chaque âge manquant par l’**âge médian des passagers qui
ont le même titre**. Les médianes sont calculées **uniquement sur le fichier d’entraînement**,
puis réutilisées pour le fichier de test, afin qu’aucune information ne provienne du jeu de
test.

```python
def extract_title(names):
    """Return the title between the comma and the period, e.g. 'Mr' or 'Miss'."""
    titles = names.str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
    titles = titles.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    return titles.where(titles.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")


age_by_title = train_raw.groupby(extract_title(train_raw["Name"]))["Age"].median()
print(age_by_title)


def impute_age(data):
    """Fill missing ages with the median age of the passenger's title group."""
    return data["Age"].fillna(extract_title(data["Name"]).map(age_by_title))


eda["Age"] = impute_age(train_raw)
print("missing ages left:", eda["Age"].isna().sum())
```

Les médianes confirment l’idée : environ 3,5 ans pour *Master*, 21 pour *Miss*, 30 pour *Mr*
et 35 pour *Mrs*. Pour utiliser cette méthode dans le modèle, remplacez
`data["Age"].fillna(-0.5)` dans `preprocess` (section E) par `impute_age(data)`. La tranche
d’âge « inconnu » disparaît alors.

</details>

In [ ]:
age_labels = ["unknown", "0–5", "5–15", "15–25", "25–40", "40–60", "60+"]
eda["AgeGroup"] = pd.cut(eda["Age"], bins=[-1, 0, 5, 15, 25, 40, 60, 120], labels=age_labels)

sns.barplot(data=eda, x="AgeGroup", y="Survived")
plt.xlabel("Age group")
plt.ylabel("Survival rate")
plt.show()

Les enfants de moins de 5 ans ont le taux de survie le plus élevé. Si vous avez conservé la
valeur provisoire −0,5, la tranche « inconnu » a survécu moins souvent que la moyenne : les
âges manquants sont plus fréquents en troisième classe.

### Variable 5 : la famille
`SibSp + Parch + 1` donne la taille de la famille qui voyage ensemble (le `+ 1` compte le
passager lui-même).

In [ ]:
eda["FamilySize"] = eda["SibSp"] + eda["Parch"] + 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=eda, x="FamilySize", y="Survived", ax=axes[0])
axes[0].set_ylabel("Survival rate")
sns.countplot(data=eda, x="FamilySize", ax=axes[1])
plt.tight_layout()
plt.show()

La plupart des passagers voyageaient seuls, et les grandes familles sont si rares que leurs
taux sont peu fiables (intervalles larges). Une variable 0/1 plus simple, *voyage seul*,
capte le motif principal : 30 % des passagers qui voyageaient seuls ont survécu, contre 51 %
des autres.

In [ ]:
eda["IsAlone"] = (eda["FamilySize"] == 1).astype(int)
sns.barplot(data=eda, x="IsAlone", y="Survived")
plt.ylabel("Survival rate")
plt.show()

### Variable 6 : `Embarked`

In [ ]:
display(eda.groupby("Embarked")["Survived"].agg(survival_rate="mean", passengers="count").round(3))

Les passagers embarqués à Cherbourg ont survécu plus souvent (55 %), sans doute parce que
beaucoup d’entre eux voyageaient en première classe. Les deux ports manquants deviendront `S`,
le plus fréquent.

### Variables 7, 8, 9 : `Fare`, `Name`, `Ticket`
Ces trois colonnes vous sont laissées.

### Exercice Q2 · Création de variables
Construisez des variables à partir de `Fare`, `Name` et `Ticket`, ajoutez-les aux entrées du
modèle, et vérifiez si l’accuracy de validation (section F) et l’accuracy en validation
croisée (section H) s’améliorent. Quelques idées :

- `Fare` : la distribution est très asymétrique ; essayez `np.log1p`. N’oubliez pas qu’un passager du fichier de test n’a pas de tarif.
- `Name` : le titre (voir Q1), encodé en colonnes one-hot.
- `Ticket` : les passagers qui partagent un numéro de billet voyageaient ensemble ; combien de personnes partagent chaque billet ?

In [ ]:
# Your turn (Q2): build features from Fare, Name and Ticket.

<details>
<summary>Réponse de référence Q2 — à ouvrir après avoir essayé</summary>

Exécutez ce code **après** la section E (il utilise `preprocess`, `FEATURES` et `y_all`).
Chaque nouvelle variable est calculée de la même façon pour les deux fichiers, et les valeurs
« apprises » (le tarif médian) proviennent uniquement du fichier d’entraînement.

```python
def extract_title(names):
    titles = names.str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
    titles = titles.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    return titles.where(titles.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")


fare_median = train_raw["Fare"].median()
# Ticket sizes use both files: a group can be split between them. No labels are used.
ticket_counts = pd.concat([train_raw["Ticket"], test_raw["Ticket"]]).value_counts()
TITLES = ["Mr", "Mrs", "Miss", "Master", "Rare"]


def preprocess_q2(raw):
    data = preprocess(raw)
    data["LogFare"] = np.log1p(raw["Fare"].fillna(fare_median))
    title = pd.Categorical(extract_title(raw["Name"]), categories=TITLES)
    data = data.join(pd.get_dummies(title, prefix="Title", dtype=int).set_index(raw.index))
    data["TicketGroupSize"] = raw["Ticket"].map(ticket_counts)
    return data


FEATURES_Q2 = FEATURES + ["LogFare", "TicketGroupSize"] + [f"Title_{t}" for t in TITLES]
train_q2 = preprocess_q2(train_raw)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
for name, cols in [("original features", FEATURES), ("with Q2 features", FEATURES_Q2)]:
    scores = cross_val_score(forest, train_q2[cols], y_all, cv=cv)
    print(f"{name:18s}: {scores.mean():.3f} ± {scores.std():.3f}")
```

Avec une forêt aléatoire, les nouvelles variables augmentent l’accuracy en validation croisée
d’environ deux points. La liste fixe des catégories de titres garantit que les deux fichiers
obtiennent les mêmes colonnes, même si un titre est absent de l’un d’eux.

</details>

## E. Prétraitement : une seule fonction pour les deux fichiers

La section D a exploré des variables sur une copie. Nous regroupons maintenant dans une
fonction les transformations que nous conservons. Appliquer **la même fonction** aux fichiers
d’entraînement et de test garantit que le modèle voit les mêmes colonnes, encodées de la même
façon, à l’entraînement comme à la prédiction.

La fonction repart chaque fois des données brutes et renvoie une nouvelle table ; elle ne
modifie jamais son entrée.

In [ ]:
AGE_BINS = [-1, 0, 5, 15, 25, 40, 60, 120]


def preprocess(raw):
    """Turn a raw Titanic table into numeric features. Returns a new DataFrame."""
    data = raw.copy()
    data["Sex"] = data["Sex"].map({"male": 1, "female": 0})
    data = data.join(pd.get_dummies(data["Pclass"], prefix="Pclass", dtype=int))
    data["Cabin"] = data["Cabin"].notna().astype(int)
    # Q1: replace the -0.5 placeholder with a better imputation.
    data["Age"] = pd.cut(data["Age"].fillna(-0.5), bins=AGE_BINS, labels=False)
    data["Embarked"] = data["Embarked"].fillna("S").map({"C": 1, "Q": 2, "S": 3})
    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
    return data


train = preprocess(train_raw)
test = preprocess(test_raw)

FEATURES = ["Sex", "Age", "Cabin", "Pclass_1", "Pclass_2", "Pclass_3", "Embarked", "IsAlone"]
# After Q2, add your new features to this list (e.g. "LogFare", "Title_Mr", ...).
TARGET = "Survived"

X_all, y_all = train[FEATURES], train[TARGET]
display(X_all.head())
print("missing values in the training features:", int(X_all.isna().sum().sum()))
print("missing values in the test features:    ", int(test[FEATURES].isna().sum().sum()))

Toutes les variables sont maintenant numériques et aucune valeur ne manque. Il reste deux
raccourcis que vous pourrez améliorer dans Q5. `Age` est désormais un numéro de tranche d’âge
de 0 à 6. `Embarked` est codée 1/2/3, ce qui suppose entre les ports un ordre qui n’existe
pas (un encodage one-hot serait plus fidèle).

### Exercice Q3 · PCA
(a) Appliquez une PCA (analyse en composantes principales) à ces variables (essayez
différents sous-ensembles) et représentez les passagers en 2D, colorés selon `Survived`.
Les deux classes se séparent-elles ?

(b) Conservez les composantes qui expliquent **90 % de la variance**, ajustez une régression
logistique sur celles-ci, et comparez son accuracy en validation croisée à celle d’une
régression logistique sur les variables d’origine.

La PCA est-elle un bon choix pour ce problème ?

In [ ]:
# Your turn (Q3): PCA in 2-D, then a logistic regression on 90% of the variance.

<details>
<summary>Réponse de référence Q3 — à ouvrir après avoir essayé</summary>

Les variables ont des échelles différentes (`Age` va de 0 à 6, les autres valent 0/1 ou 1–3) :
nous les standardisons donc avant la PCA ; sinon, la variable la plus étendue domine les
composantes.

```python
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# (a) 2-D view
coords = make_pipeline(StandardScaler(), PCA(n_components=2)).fit_transform(X_all)
plt.scatter(coords[:, 0], coords[:, 1], c=y_all, cmap="coolwarm", alpha=0.6)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("Passengers on the first two components (red = survived)")
plt.show()

# (b) logistic regression with and without PCA
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
with_pca = make_pipeline(StandardScaler(), PCA(n_components=0.90), LogisticRegression(max_iter=1000))
without_pca = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
for name, model in [("with PCA (90%)", with_pca), ("without PCA", without_pca)]:
    scores = cross_val_score(model, X_all, y_all, cv=cv)
    print(f"{name:15s}: {scores.mean():.3f} ± {scores.std():.3f}")
print("components kept:", with_pca.fit(X_all, y_all).named_steps["pca"].n_components_)
```

Le graphique 2D montre des groupes de points qui se chevauchent (les variables ne prennent que
quelques valeurs), avec des survivants concentrés dans certains groupes. Conserver 90 % de la
variance nécessite la plupart des composantes, et l’accuracy est à peu près la même que sans
PCA. La PCA n’est pas très utile ici : il n’y a que 8 variables, donc peu de choses à
compresser, et les composantes sont plus difficiles à interpréter que « sexe » ou « classe ».
La PCA devient rentable avec de nombreuses variables corrélées, comme les 64 pixels du
notebook 01.

</details>

## F. Un premier modèle

Comme dans le notebook 01, nous mettons de côté une partie du fichier d’entraînement pour
**valider** nos choix : 70 % pour entraîner, 30 % pour valider. `stratify=y_all` conserve le
même taux de survie dans les deux parties, et `random_state` rend la séparation reproductible.

(Le fichier de test de Kaggle ne peut pas jouer ce rôle, puisque nous ne connaissons pas ses
réponses.)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.3, stratify=y_all, random_state=76
)

first_tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
print(f"decision tree, validation accuracy: {first_tree.score(X_val, y_val):.3f}")

Près de 80 %, bien au-dessus de la baseline de 62 %. D’autres modèles peuvent-ils faire mieux ?

## G. Comparer des classifieurs

Nous entraînons huit classifieurs avec des réglages raisonnables et comparons leur accuracy
sur la partie d’entraînement et sur la partie de validation.

| Modèle | L’idée en une ligne |
|---|---|
| Arbre de décision | Une suite de questions oui/non sur les variables |
| Régression logistique | Une somme pondérée des variables, transformée en probabilité |
| k plus proches voisins | Prédire comme les *k* passagers d’entraînement les plus semblables |
| SVM | La frontière qui laisse la plus grande marge entre les classes |
| Perceptron | Le classifieur linéaire le plus simple, ancêtre des réseaux de neurones |
| Forêt aléatoire | De nombreux arbres sur des échantillons bootstrap ; vote majoritaire |
| Arbres extrêmement aléatoires (*Extra trees*) | Comme une forêt aléatoire, avec des coupures choisies de façon plus aléatoire |
| AdaBoost | De nombreux petits arbres, chacun se concentrant sur les erreurs des précédents |

`class_weight="balanced"` donne plus de poids à la classe la moins fréquente (les survivants)
pendant l’entraînement. `random_state` rend reproductibles les modèles qui comportent une part
d’aléatoire.

In [ ]:
models = {
    "Decision tree": DecisionTreeClassifier(class_weight="balanced", min_weight_fraction_leaf=0.01, random_state=0),
    "Logistic regression": LogisticRegression(max_iter=1000),
    "k-nearest neighbours": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(random_state=0),
    "Perceptron": Perceptron(class_weight="balanced", random_state=0),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=0),
    "Extra trees": ExtraTreesClassifier(n_estimators=300, max_features="sqrt", bootstrap=True,
                                        min_samples_split=10, random_state=0),
    "AdaBoost": AdaBoostClassifier(n_estimators=400, learning_rate=0.1, random_state=0),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    rows.append({
        "Classifier": name,
        "Training accuracy (%)": round(100 * model.score(X_train, y_train), 1),
        "Validation accuracy (%)": round(100 * model.score(X_val, y_val), 1),
    })

results = pd.DataFrame(rows).sort_values("Validation accuracy (%)", ascending=False)
display(results)

Comment lire ce tableau :

- Tous les modèles obtiennent entre 78 % et 82 % en validation. Avec 268 passagers de
  validation, un passager représente 0,4 point : des écarts d’un ou deux points restent donc
  dans les limites du bruit mesuré dans les notebooks 01 et 02. Ne concluez pas que le modèle
  en tête est le meilleur.
- Un modèle bien meilleur à l’entraînement qu’en validation fait du **surapprentissage** : il a
  mémorisé des détails des passagers d’entraînement qui ne se généralisent pas. Comparez les
  deux colonnes de la forêt aléatoire.
- Les k plus proches voisins, le SVM, la régression logistique et le perceptron sont sensibles
  à l’**échelle** des variables. Les nôtres sont de petits entiers, donc cela compte peu ici,
  mais les standardiser est une bonne habitude (un indice pour Q5).

## H. Méthodes d’ensemble et validation croisée

### Vote
Un `VotingClassifier` interroge plusieurs modèles et renvoie le **vote majoritaire**
(`voting="hard"`). Lorsque les modèles font des erreurs différentes, le vote peut obtenir une meilleure accuracy que chacun d’eux. `clone` crée des copies neuves, non entraînées, de nos modèles.

In [ ]:
voting = VotingClassifier(
    estimators=[
        ("knn", clone(models["k-nearest neighbours"])),
        ("svm", clone(models["SVM"])),
        ("tree", clone(models["Decision tree"])),
        ("extra", clone(models["Extra trees"])),
        ("logistic", clone(models["Logistic regression"])),
        ("adaboost", clone(models["AdaBoost"])),
    ],
    voting="hard",
)
voting.fit(X_train, y_train)
print(f"voting ensemble, validation accuracy: {voting.score(X_val, y_val):.3f}")

### Une estimation plus fiable
Le score de validation ci-dessus dépend d’une séparation particulière. La validation croisée
sur l’ensemble du fichier d’entraînement donne une estimation plus stable (notebook 01).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = cross_val_score(voting, X_all, y_all, cv=cv, scoring="accuracy")
print("fold accuracies:", np.round(cv_scores, 3))
print(f"voting ensemble, cross-validated accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

L’accuracy en validation croisée est inférieure au score de validation unique obtenu plus
haut : cette séparation était un peu chanceuse. **Rapportez le chiffre obtenu en validation
croisée.**

## I. Prédire pour Kaggle

Pour les prédictions finales, nous réentraînons l’ensemble par vote sur la **totalité** des
891 passagers d’entraînement, puis nous prédisons les 418 passagers de test. Kaggle attend un
CSV avec exactement deux colonnes, `PassengerId` et `Survived`.

In [ ]:
voting.fit(X_all, y_all)
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": voting.predict(test[FEATURES]),
})

output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / "titanic_submission.csv"
submission.to_csv(submission_path, index=False)

print(submission.shape, "→", submission_path)
display(submission.head())
print("predicted survival rate:", round(submission["Survived"].mean(), 3))

**La soumission est facultative.** Pour obtenir un score, créez un compte Kaggle gratuit,
rejoignez la [compétition Titanic](https://www.kaggle.com/c/titanic), et téléversez ce fichier
sous *Submit Prediction*. Lors des éditions précédentes de ce laboratoire, une soumission
semblable obtenait environ 76 %. Kaggle évalue un autre ensemble de passagers : attendez-vous
à ce que votre score diffère quelque peu de votre accuracy en validation croisée.

### Exercice Q4 · Recherche aléatoire
Choisissez l’un des algorithmes ci-dessus et utilisez `RandomizedSearchCV` (notebook 01) pour
trouver de bons hyperparamètres. Utilisez la validation croisée sur `X_all, y_all`. De combien
l’accuracy en validation croisée s’améliore-t-elle par rapport aux réglages par défaut ?

In [ ]:
# Your turn (Q4): tune one model with RandomizedSearchCV.

<details>
<summary>Réponse de référence Q4 — à ouvrir après avoir essayé</summary>

Une forêt aléatoire a plusieurs hyperparamètres qui interagissent, ce qui se prête bien à la
recherche aléatoire.

```python
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "n_estimators": randint(100, 500),
    "max_depth": [3, 4, 5, 6, 8, 10, None],
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", None],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=0), param_distributions,
    n_iter=20, cv=cv, scoring="accuracy", random_state=0, n_jobs=-1,
)
search.fit(X_all, y_all)

default_scores = cross_val_score(RandomForestClassifier(random_state=0), X_all, y_all, cv=cv)
print(f"default random forest: {default_scores.mean():.3f}")
print(f"tuned random forest:   {search.best_score_:.3f}")
print("best parameters:", search.best_params_)
```

La forêt ajustée préfère des arbres moins profonds ou des feuilles plus grandes que les
réglages par défaut, qui font du surapprentissage. Le gain est d’un point ou deux : l’ajustement
aide, mais de meilleures **variables** (Q2, Q5) aident généralement davantage. `best_score_` est
légèrement optimiste, car c’est le meilleur de 20 essais sur les mêmes plis.

</details>

### Exercice Q5 · Faire mieux
Utilisez tout ce que vous avez vu dans cette démo (création de variables, encodage, mise à
l’échelle, choix du modèle, ajustement des hyperparamètres) pour améliorer autant que possible
l’accuracy en validation croisée. Si vous soumettez à Kaggle, visez **80 % ou plus** au
classement public (*leaderboard*).

Indices :
1. Utilisez l’encodage one-hot pour les variables catégorielles (`Embarked`, titre, pont…).
2. Standardisez les variables pour les modèles sensibles à l’échelle.
3. Vérifiez quelles variables sont les plus liées à la cible, et retirez celles qui sont inutiles.
4. Utilisez vos connaissances du domaine pour extraire de l’information (titres, groupes familiaux, tarif par personne…).
5. Essayez l’empilement (*stacking*) : `StackingClassifier` entraîne un modèle sur les prédictions d’autres modèles.

Placez le prétraitement dans un `Pipeline` avec un `ColumnTransformer`, pour que la validation
croisée reste honnête (notebook 01, section E).

In [ ]:
# Your turn (Q5): build the best model you can and report its cross-validated accuracy.

<details>
<summary>Réponse de référence Q5 (une approche possible) — à ouvrir après avoir essayé</summary>

Conservez quelques variables lisibles, imputez l’âge selon le titre, encodez les catégories en
one-hot, et laissez un `ColumnTransformer` standardiser les colonnes numériques dans un
pipeline.

```python
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def extract_title(names):
    titles = names.str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
    titles = titles.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    return titles.where(titles.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")


age_by_title = train_raw.groupby(extract_title(train_raw["Name"]))["Age"].median()
fare_median = train_raw["Fare"].median()
ticket_counts = pd.concat([train_raw["Ticket"], test_raw["Ticket"]]).value_counts()


def build_features(raw):
    data = pd.DataFrame(index=raw.index)
    data["Title"] = extract_title(raw["Name"])
    data["Pclass"] = raw["Pclass"].astype(str)
    data["Sex"] = raw["Sex"]
    data["Embarked"] = raw["Embarked"].fillna("S")
    data["Age"] = raw["Age"].fillna(data["Title"].map(age_by_title))
    data["LogFare"] = np.log1p(raw["Fare"].fillna(fare_median))
    data["FamilySize"] = raw["SibSp"] + raw["Parch"] + 1
    data["TicketGroupSize"] = raw["Ticket"].map(ticket_counts)
    data["HasCabin"] = raw["Cabin"].notna().astype(int)
    return data


categorical = ["Title", "Pclass", "Sex", "Embarked"]
numeric = ["Age", "LogFare", "FamilySize", "TicketGroupSize", "HasCabin"]
encode = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numbers", StandardScaler(), numeric),
])

X_q5, X_q5_test = build_features(train_raw), build_features(test_raw)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
candidates = {
    "logistic regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(random_state=0),
    "gradient boosting": GradientBoostingClassifier(max_depth=3, n_estimators=100, random_state=0),
}
for name, model in candidates.items():
    scores = cross_val_score(make_pipeline(encode, model), X_q5, y_all, cv=cv)
    print(f"{name:20s}: {scores.mean():.3f} ± {scores.std():.3f}")
```

Ces pipelines atteignent environ 82 à 83 % en validation croisée, contre environ 79 % pour
l’ensemble par vote de la section H. Pour soumettre, ajustez le meilleur pipeline sur
`X_q5, y_all` et prédisez `X_q5_test`. Sur cette compétition, les scores du classement public
sont généralement inférieurs de quelques points à l’accuracy en validation croisée : 80 % sur
Kaggle est donc un objectif exigeant.

</details>

### Exercice Q6 · À quel point sommes-nous sûrs ?
Pour l’ensemble par vote final, estimez un **intervalle de confiance bootstrap à 95 %** pour
l’accuracy, en utilisant uniquement le fichier d’entraînement (notebook 02 : entraîner sur un
échantillon bootstrap, évaluer sur les passagers hors sac (OOB)). Si vous avez soumis à
Kaggle, votre score au classement public tombe-t-il dans l’intervalle ?

In [ ]:
# Your turn (Q6): bootstrap confidence interval for the voting ensemble.

<details>
<summary>Réponse de référence Q6 — à ouvrir après avoir essayé</summary>

100 répétitions limitent le temps d’exécution ; davantage de répétitions donnent un
intervalle plus stable.

```python
rng = np.random.default_rng(0)
n = len(y_all)
boot_scores = []
for _ in range(100):
    idx = rng.choice(n, size=n, replace=True)
    oob = np.setdiff1d(np.arange(n), idx)
    model = clone(voting).fit(X_all.iloc[idx], y_all.iloc[idx])
    boot_scores.append(accuracy_score(y_all.iloc[oob], model.predict(X_all.iloc[oob])))

low, high = np.percentile(boot_scores, [2.5, 97.5])
print(f"mean OOB accuracy: {np.mean(boot_scores):.3f}")
print(f"95% bootstrap CI: [{low:.1%}, {high:.1%}]")
```

L’intervalle va d’environ 76 % à 83 %. Un score Kaggle d’environ 76 à 78 % se situe près de
sa borne inférieure. Deux raisons expliquent un score un peu plus bas au classement public :
les passagers de test forment un échantillon différent, et nos choix de modélisation
(variables, modèles) ont été faits en regardant les données d’entraînement, ce qui rend un peu
optimistes les estimations fondées sur l’entraînement.

</details>

## À retenir

- Commencez par la **question** et les **données** : la cible, les variables, les valeurs manquantes, et une baseline simple à battre.
- L’exploration guide la **création de variables** : ici, les signaux les plus forts sont le sexe, la classe, la tranche d’âge et le fait qu’une cabine soit enregistrée ou non.
- Placez les transformations dans **une seule fonction** (ou mieux, dans un pipeline) et appliquez-la à chaque fichier.
- Comparez les modèles sur des données de validation, puis rapportez un score en **validation croisée**, idéalement accompagné d’un **intervalle de confiance**.
- Les gains viennent généralement davantage de meilleures variables que de modèles plus complexes.

## Références

1. [Compétition Titanic sur Kaggle](https://www.kaggle.com/c/titanic)
2. [Predicting the survival of Titanic passengers](https://towardsdatascience.com/predicting-the-survival-of-titanic-passengers-30870ccc7e8), Towards Data Science
3. [Titanic survival dataset: exploratory data analysis](https://becominghuman.ai/titanic-survival-dataset-part-1-2-exploratory-data-analysis-5b98f7917913), Becoming Human
4. [How to (almost) win Kaggle competitions](https://yanirseroussi.com/2014/08/24/how-to-almost-win-kaggle-competitions/), Yanir Seroussi
5. [Kaggle tips and tricks](https://www.dataquest.io/blog/kaggle-tips-tricks/), Dataquest
6. Guide d’utilisation de scikit-learn : [méthodes d’ensemble](https://scikit-learn.org/stable/modules/ensemble.html) et [`ColumnTransformer`](https://scikit-learn.org/stable/modules/compose.html#column-transformer)